![Week 5 Day 5](../assets/banners/w5d5.png)

# Week 5 Exercise: A Moderation Second-Grader for Borderline Marks

Emma's exam board has a rule that predates AI by about a century: **no single examiner is trusted on their own.** After the first marking, a *moderator* re-marks a sample of scripts. Where the two agree, the mark stands. Where they disagree by more than the tolerance, a senior examiner decides.

GradeWise can do the same. Most marks are clear-cut: full marks, or a blank. The **borderline** ones in the middle are where two examiners (human or AI) most often disagree. In this exercise you'll add a **second, independent grader** that re-marks only the borderline answers, agrees a mark when the two are close, and escalates to Emma when they aren't. Then you'll use Day 5's golden set to prove whether it's worth the extra cost.

### 🎯 What you'll build
1. `is_borderline(result, question)`: which marks deserve a second opinion?
2. `moderate(first, second, question)`: agree a mark, or escalate to Emma
3. A **moderation pass** over the class that re-marks only the borderline answers
4. A before/after comparison on the golden set: agreement (QWK, MAE, exact), **cost** (extra grading calls) and **workload** (answers in Emma's queue)
5. A recommendation, and (bonus) a sketch of where moderation would live in the orchestrator

Cells marked `# ✋ TODO` are yours; everything else is given. The notebook runs **as-is in dry-run mode** (two stand-in graders, no key, no cost), so you can build and debug the whole thing for free, then switch to real models.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Real runs cost a little: with the default <code>MODEL_MODE=cheap</code> (<code>gpt-6-luna</code>, $0.10 / $0.50 per million tokens) grading all ten students once is well under a dollar, plus the moderator's calls for the borderline answers. Real mode grades only <code>REAL_STUDENTS = 3</code> students by default (2 with <code>DATA_SIZE=small</code>), so it's a few cents. <code>MODEL_MODE=free</code> costs $0, but OpenRouter's free models allow 20 requests a minute and 50 a day (use <code>DATA_SIZE=small</code>), and the model changes per call, so the agreement numbers wobble. Build and test everything in dry-run mode first.</span>
        </td>
    </tr>
</table>

💡 **Fast & cheap run.** Already know the ideas and just want to watch everything run? Put `DATA_SIZE=small` in your `.env` (2 PDFs · Q1–Q3 · 2 students) and keep the default `MODEL_MODE=cheap` (`openai/gpt-6-luna` via OpenRouter, a few cents of API calls). Or set `MODEL_MODE=free` to run on OpenRouter's free models for $0, with the same `OPENROUTER_API_KEY` and no credits: it's rate-limited to 20 requests a minute and 50 a day (1,000 a day once you've ever bought $10 of credits), so pair it with `DATA_SIZE=small`; the model changes from call to call, so results vary; and free providers may log or train on prompts, which is fine for this course's fictional data and never for real students' work. Restart the kernel after editing `.env`; the setup cell below prints what you're running on.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 💡 MODEL_MODE=cheap (default) and MODEL_MODE=free both need just OPENROUTER_API_KEY
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"❌ {key_name} not set. Most of today's cells still run offline; the few that call a model will")
        print("   show a friendly 'Missing API key' panel. See Part 5 of the setup guide, or setup/SWAP_PROVIDER.md")

# 📦 What this run uses: MODEL_MODE and DATA_SIZE in .env (restart the kernel after changing them)
print(settings.describe_models())
print(datasets.describe())

In [ ]:
import json

from langchain_core.embeddings import DeterministicFakeEmbedding

from gradewise import datasets
from gradewise.config import DATA_DIR, OUTPUT_DIR, settings
from gradewise.console import success, warn
from gradewise.evals.compat import import_pandas
from gradewise.ingestion import ingest
from gradewise.rag import get_retriever
from gradewise.schemas import ExamPaper, GradeResult, ReportCard

pd = import_pandas()  # 💡 = `import pandas as pd`, with a fix for locked-down Windows machines

EXAM_FILE = DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json"
FULL_EXAM = ExamPaper.model_validate_json(EXAM_FILE.read_text(encoding="utf-8"))
QUESTIONS = {q.id: q for q in FULL_EXAM.questions}  # every question, for the one-answer demos
TRUTH = json.loads((DATA_DIR / "answer-sheets" / "synthetic" / "transcripts.json").read_text(encoding="utf-8"))

# 📦 What the batch runs grade: all 15 parts and 10 students, or Q1–Q3 and 2 students with DATA_SIZE=small
exam = datasets.subset_exam(FULL_EXAM)
STUDENTS = datasets.synthetic_students()

# 🆓 A keyword-only (BM25) copy of the CS201 notes: fake embeddings, real keyword search, zero cost
OFFLINE_DB = OUTPUT_DIR / "week5_offline_db"
fake_embeddings = DeterministicFakeEmbedding(size=64)
ingest(
    collection="cs201_offline",
    embeddings=fake_embeddings,
    persist_dir=OFFLINE_DB,
    quiet=True,
    files=datasets.kb_files(),
)  # 📦 all 4 PDFs, or unit1 + the handbook with DATA_SIZE=small
bm25 = get_retriever("cs201_offline", k=4, mode="bm25", embeddings=fake_embeddings, persist_dir=OFFLINE_DB)

success(
    "Ready",
    {
        "Exam": f"{exam.title} ({len(exam.questions)} questions, {exam.total_marks:g} marks)",
        "Students": f"{len(STUDENTS)} synthetic scripts",
        "Offline KB": f"cs201_offline (BM25, {len(datasets.kb_files())} PDFs)",
        "Grading model": settings.grading_model,
    },
)

## Part 1: The two examiners

The first marker is GradeWise's grading model (`gradewise.models.grading_model()`). The moderator should be **independent**: a different model makes different mistakes, which is the whole point of a second opinion. We use `MODERATOR_MODEL` from `.env` if you set one (e.g. `openrouter:openai/gpt-6.1-terra`, or any other OpenRouter model ID), or else your everyday chat model. ⚠️ In `MODEL_MODE=cheap` both default to `gpt-6-luna` (the grader just thinks harder), so set `MODERATOR_MODEL` for a truly independent second opinion. As always, no hard-coded model names.

In [ ]:
from gradewise.errors import MissingAPIKeyError
from gradewise.models import grading_model, make_chat_model

MODERATOR = os.getenv("MODERATOR_MODEL") or settings.chat_model
try:
    first_llm, moderator_llm = grading_model(), make_chat_model(MODERATOR)
    DRY_RUN = False
except MissingAPIKeyError as err:
    DRY_RUN = True
    warn(f"Dry run: {err.what} Using two stand-in graders, so everything below is free.")
print(f"first marker = {settings.grading_model}\nmoderator    = {MODERATOR}\nDRY_RUN      = {DRY_RUN}")

### 🎭 Dry-run stand-ins
In dry-run mode the first marker is Day 1's `KeywordGraderModel` (harsh: it needs the rubric's exact words). The moderator, `SecondOpinionGrader`, is a stand-in with a *different* blind spot: it matches keyword **stems** (the first five letters), so it forgives spelling and word forms ("exclution", "atomically"), as principle 1 asks. Two keyword matchers are still not examiners, but they disagree in realistic ways, which gives your moderation logic something to work on.

In [ ]:
import re

from langchain_core.messages import HumanMessage

from gradewise.grading.fakes import KeywordGraderModel, tool_call_message


class SecondOpinionGrader(KeywordGraderModel):
    """🎭 STAND-IN moderator: like KeywordGraderModel, but matches keyword stems, and needs fewer of them."""

    def _respond(self, messages):
        message = super()._respond(messages)
        if not message.tool_calls or message.tool_calls[0]["name"] != "GradeResult":
            return message
        prompt = next(str(m.content) for m in messages if isinstance(m, HumanMessage) and "<rubric>" in str(m.content))
        rubric = json.loads(prompt.split("<rubric>")[1].split("</rubric>")[0])
        answer = prompt.split("<student_answer>")[1].split("</student_answer>")[0].lower()
        stems = {w[:5] for w in re.findall(r"[a-z0-9]+", answer)}
        args = dict(message.tool_calls[0]["args"])
        criteria = []
        for given, c in zip(args["criteria"], rubric["criteria"], strict=True):
            keywords = [k.lower() for k in c["keywords"]]
            hits = [k for k in keywords if all(w[:5] in stems for w in re.findall(r"[a-z0-9]+", k))]
            share = len(hits) / len(keywords) if keywords else 0
            criteria.append({**given, "awarded": c["marks"] if share >= 0.25 else (c["marks"] / 2 if hits else 0.0)})
        args.update(criteria=criteria, awarded_marks=sum(c["awarded"] for c in criteria))
        return tool_call_message("GradeResult", args)


if DRY_RUN:
    first_llm, moderator_llm = KeywordGraderModel(), SecondOpinionGrader()
REAL_STUDENTS = 3
students = STUDENTS if DRY_RUN else STUDENTS[:REAL_STUDENTS]  # 📦 STUDENTS honours DATA_SIZE

## Part 2: First marking, and the starting point

Grade the class with the first marker (Day 2's grading graph), and measure it against the golden set with Day 5's `compare_marks()`. That's the number moderation has to improve on.

In [ ]:
from gradewise.evals import compare_marks
from gradewise.evals.grading_eval import human_marks, load_golden, predicted_marks
from gradewise.grading import build_grading_graph, grade_sheet
from gradewise.orchestrator import sheets_from_transcripts

sheets = {s.student_id: s for s in sheets_from_transcripts(exam, students=students)}
first_graph = build_grading_graph(first_llm, bm25)
first_reports = {sid: grade_sheet(exam, sheet, graph=first_graph) for sid, sheet in sheets.items()}

human, _ = human_marks(datasets.golden_rows(load_golden()))  # 📦 all of it, or 2 students × Q1–Q3
before = compare_marks(human, predicted_marks(first_reports.values()))["overall"]
print("first marker alone:", before)

## Part 3: What counts as borderline?

Re-marking everything doubles the cost. Re-marking nothing gains nothing. You need a rule. Real exam boards moderate scripts **near a grade boundary** and marks in the **middle of the range**, where judgement matters most; you also have the grader's own **confidence**.

In [ ]:
def is_borderline(result: GradeResult, question) -> bool:
    """True if this mark deserves a second opinion from the moderator.

    ✋ TODO: decide what "borderline" means. Some ideas (combine a few):
       - partial credit in the middle of the range (e.g. between 25% and 75% of the question's marks)
       - low grader confidence (e.g. below 0.8)
       - already flagged for review? Then Emma will see it anyway: no need to pay for a moderator
    """
    return False  # ✋ TODO: replace this line

In [ ]:
borderline = [
    (sid, r.question_id)
    for sid, rep in first_reports.items()
    for r in rep.results
    if is_borderline(r, QUESTIONS[r.question_id])
]
total_answers = sum(len(rep.results) for rep in first_reports.values())
share = len(borderline) / total_answers
print(f"{len(borderline)} of {total_answers} answers are borderline ({share:.0%} extra grading calls)")
pd.Series([qid for _, qid in borderline]).value_counts().rename("borderline answers").to_frame().T

## Part 4: The moderator, and agreeing a mark

The moderator re-marks one answer with `grade_answer()`, completely independently: it never sees the first mark (an examiner who sees the first mark just agrees with it: that's *anchoring*). Then `moderate()` combines the two:

- **within tolerance** (say, 1 mark apart): agree a mark, criterion by criterion
- **beyond tolerance**: keep the first mark, but flag it for Emma with both marks in the reason

`half_up()` rounds to the nearest half mark, rounding halves **up**, like Day 5's `round_marks()`.

In [ ]:
import math

from gradewise.grading import grade_answer


def half_up(x: float) -> float:
    """Round to the nearest half mark, halves up: 0.75 → 1.0, 0.25 → 0.5."""
    return math.floor(x * 2 + 0.5) / 2


def second_opinion(student_id: str, question_id: str) -> GradeResult:
    """The moderator marks one answer from scratch (it never sees the first mark)."""
    answer = sheets[student_id].answer_for(question_id)
    return grade_answer(QUESTIONS[question_id], answer, llm=moderator_llm, retriever=bm25, exam=exam)

In [ ]:
def moderate(first: GradeResult, second: GradeResult, question, tolerance: float = 1.0) -> GradeResult:
    """Combine the first marker's grade with the moderator's.

    ✋ TODO:
       - if the two totals differ by MORE than `tolerance` marks: keep the first mark, but set
         needs_review=True with a review_reason that shows both marks (Emma decides)
       - otherwise: agree a mark. For each criterion, take the mean of the two marks rounded to the
         nearest half mark (use half_up), then call .clamp(caps) so the total is recomputed and capped
    """
    return first  # ✋ TODO: replace this line

A few quick self-checks. Run them until they all pass:

In [ ]:
def fake_grade(*marks: float, confidence: float = 0.85) -> GradeResult:
    criteria = [{"criterion_id": f"Q3.c{i}", "awarded": m, "justification": "…"} for i, m in enumerate(marks, 1)]
    return GradeResult(
        question_id="Q3", awarded_marks=sum(marks), max_marks=4, criteria=criteria, feedback="…", confidence=confidence
    )


q3 = QUESTIONS["Q3"]
close = moderate(fake_grade(1, 1, 0), fake_grade(2, 1, 0), q3)
far = moderate(fake_grade(0, 0, 0), fake_grade(2, 1, 1), q3)
capped = moderate(fake_grade(2, 1, 1), fake_grade(2, 1, 1), q3)
checks = [
    (close.awarded_marks == 2.5 and not close.needs_review, "1 mark apart → agree: (1+2)/2 = 1.5 on c1, total 2.5"),
    (far.awarded_marks == 0 and far.needs_review and "4" in (far.review_reason or ""), "4 marks apart → escalate"),
    (capped.awarded_marks == 4 <= q3.max_marks, "never above the question's maximum"),
]
for ok, what in checks:
    print(("✅ " if ok else "✋ not yet: ") + what)

## Part 5: The moderation pass

Now run it over the class: re-mark each borderline answer, moderate it, and rebuild each student's `ReportCard` with `ReportCard.from_results()` (so totals and grade bands stay right).

In [ ]:
moderated_reports, log = {}, []
for sid, report in first_reports.items():
    results = []
    for r in report.results:
        if (sid, r.question_id) in borderline:
            second = second_opinion(sid, r.question_id)
            agreed = moderate(r, second, QUESTIONS[r.question_id])
            escalated = agreed.needs_review and not r.needs_review
            log.append(
                {
                    "student": sid,
                    "question": r.question_id,
                    "first": r.awarded_marks,
                    "moderator": second.awarded_marks,
                    "agreed": agreed.awarded_marks,
                    "escalated": escalated,
                    "human": human[(sid, r.question_id)],
                }
            )
            r = agreed
        results.append(r)
    moderated_reports[sid] = ReportCard.from_results(sid, exam.exam_id, results)

log = pd.DataFrame(log, columns=["student", "question", "first", "moderator", "agreed", "escalated", "human"])
log["escalated"] = log["escalated"].astype(bool)
print(f"re-marked {len(log)} answers · escalated {int(log.escalated.sum())} to Emma")
log.head(10)

## Part 6: Was it worth it?

⚠️ With `DATA_SIZE=small` this comparison rests on 8 answers: the numbers are noise, only good for checking that the code runs. Judge moderation on the full golden set.

Three versions of the marks, compared with the human examiner:
1. **first marker alone**
2. **moderated**: agreed marks where the two were close, first marks (flagged) where they weren't
3. **moderated + Emma**: as 2, but with the escalated answers settled by Emma. We use the golden mark as a stand-in for her decision.

In [ ]:
moderated = predicted_marks(moderated_reports.values())
with_emma = dict(moderated)
for row in log[log.escalated].itertuples():
    with_emma[(row.student, row.question)] = human[(row.student, row.question)]  # 🧑‍🏫 Emma settles it

table = pd.DataFrame(
    {
        "first marker alone": before,
        "moderated": compare_marks(human, moderated)["overall"],
        "moderated + Emma": compare_marks(human, with_emma)["overall"],
    }
).T
table["extra grading calls"] = [0, len(log), len(log)]
table["answers for Emma"] = [0, int(log.escalated.sum()), int(log.escalated.sum())]
table

### ✋ TODO: your recommendation for Emma

Write one paragraph: should GradeWise moderate borderline marks? Use your numbers: how much did agreement with the human examiner improve, how many extra grading calls did it cost, and how many answers now land in Emma's queue? Would you change the borderline rule or the tolerance?

*(your answer here)*

## Part 7 (bonus): Where moderation would live in GradeWise

In the orchestrator, moderation is one more node between `aggregate` and the human-review decision. Every escalation sets `needs_review`, so Emma's existing review queue (Day 4) handles it with no extra code:

![Diagram](../assets/diagrams/386036ccc186.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  G[grade × students] --> A[aggregate] --> M[moderate<br/>borderline only] --> D{flags?}
  D -- yes --> H[[human_review]] --> W[write_report]
  D -- no --> W
```

</details>


✋ **Bonus TODO:** sketch `moderate_node(state)` for `gradewise/orchestrator.py`: for each student in `state["graded"]`, re-mark the borderline answers, and return `{"graded": {student_id: moderated_report.model_dump(mode="json")}}`. Why is the `merge_by_student` reducer exactly what you want here? (Don't edit the package file: write it in a cell, or in your own copy.)

## Recap

You added a second examiner to GradeWise, the way exam boards have done for a century:
- a **rule** for which marks are borderline, so moderation costs a fraction of a second full marking
- a **moderator** that marks independently, and a `moderate()` that agrees a mark or escalates
- an honest **before/after** on the golden set, weighing agreement against cost and Emma's workload

Next week you'll ship GradeWise as an app, and the review queue you've been feeding becomes a real screen in Emma's dashboard.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/thankyou.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ec4899;">Thank you</h2>
            <span style="color:#ec4899;">Great work! Moderation is a genuinely useful feature: if you run it with real models and find a borderline rule that beats the baseline, share your numbers (no API keys!) in <code>community_contributions/</code>.</span>
        </td>
    </tr>
</table>